# Many Models Forecasting Demo

This notebook showcases how to run MMF with local models on multiple univariate time series of monthly resolution. We will use [M4 competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128#sec5) data. The descriptions here are mostly the same as the case with the [daily resolution](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/daily/local_univariate_daily.ipynb), so we will skip the redundant parts and focus only on the essentials.

### Cluster setup

We recommend using a cluster with [Databricks Runtime 17.3 LTS for ML](https://docs.databricks.com/en/release-notes/runtime/17.3lts-ml.html). The cluster can be either a single-node or multi-node CPU cluster. Make sure to set the following Spark configurations before you start your cluster: [`spark.sql.execution.arrow.enabled true`](https://spark.apache.org/docs/3.0.1/sql-pyspark-pandas-with-arrow.html#enabling-for-conversion-tofrom-pandas) and [`spark.sql.adaptive.enabled false`](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution). You can do this by specifying [Spark configuration](https://docs.databricks.com/en/compute/configure.html#spark-configuration) in the advanced options on the cluster creation page.

### Install and import packages

In [0]:
%pip install -r ../../requirements-local.txt --quiet
%pip install datasetsforecast==0.0.8 --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from tqdm.autonotebook import tqdm

/root/.ipykernel/2099/command-8771884748425605-987820437:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [0]:
import pathlib
import pandas as pd
from datasetsforecast.m4 import M4
from mmf_sa import run_forecast

/databricks/python_shell/lib/lsp_backend/document_sync_manager.py:81: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  sanitized = [sanitize_command_magics(command) for command, _ in commands]
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


### Prepare data 
We are using [`datasetsforecast`](https://github.com/Nixtla/datasetsforecast/tree/main/) package to download M4 data.

In [0]:
# Number of time series
n = 100


def create_m4_monthly():
    y_df, _, _ = M4.load(directory=str(pathlib.Path.home()), group="Monthly")
    target_ids = {f"M{i}" for i in range(1, n)}
    y_df = y_df[y_df["unique_id"].isin(target_ids)]
    y_df = (
        y_df.groupby("unique_id", group_keys=False)
             .apply(lambda g: transform_group(g, g.name))
             .reset_index(drop=True)
    )
    return y_df


def transform_group(df, unique_id):
    if len(df) > 60:
        df = df.iloc[-60:]
    start = pd.Timestamp("2018-01-01")
    date_idx = pd.date_range(start=start, periods=len(df), freq="ME", name="ds")
    res_df = pd.DataFrame({
        "ds": date_idx,
        "unique_id": unique_id,
        "y": df["y"].to_numpy()
    })
    return res_df

We are going to save this data in a delta lake table. Provide catalog and database names where you want to store the data.

In [0]:
catalog = "mmf_demo_aym"  # Name of the catalog we use to manage our assets
db = "m4_advanced"  # Name of the schema we use to manage our assets (e.g. datasets)
user = spark.sql('select current_user() as user').collect()[0]['user'] # User email address

In [0]:
# Making sure that the catalog and the schema exist
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

(
    spark.createDataFrame(create_m4_monthly())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.m4_monthly_train")
)

91.7MiB [00:01, 86.0MiB/s]
ERROR:datasetsforecast.utils:ERROR, something went wrong downloading data
INFO:datasetsforecast.utils:Successfully downloaded Monthly-train.csv, 91655432, bytes.
7.94MiB [00:00, 67.7MiB/s]
ERROR:datasetsforecast.utils:ERROR, something went wrong downloading data
INFO:datasetsforecast.utils:Successfully downloaded Monthly-test.csv, 7942698, bytes.
4.34MiB [00:00, 110MiB/s]                  
ERROR:datasetsforecast.utils:ERROR, something went wrong downloading data
INFO:datasetsforecast.utils:Successfully downloaded M4-info.csv, 4335598, bytes.
100%|██████████| 3.56M/3.56M [00:00<00:00, 61.2MiB/s]
INFO:datasetsforecast.utils:Successfully downloaded submission-Naive2.zip, 3564691, bytes.
INFO:datasetsforecast.utils:Decompressing zip file...
INFO:datasetsforecast.utils:Successfully decompressed /root/m4/datasets/submission-Naive2.zip
/root/.ipykernel/2099/command-8771884748425608-2549846427:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping co

Let's take a peak at the dataset:

In [0]:
display(
  spark.sql(f"select unique_id, count(ds) as count from {catalog}.{db}.m4_monthly_train group by unique_id order by unique_id")
  )

unique_id,count
M1,60
M10,60
M11,60
M12,60
M13,60
M14,60
M15,60
M16,60
M17,60
M18,60


In [0]:
display(
  spark.sql(f"select * from {catalog}.{db}.m4_monthly_train where unique_id in ('M1', 'M2', 'M3', 'M4', 'M5') order by unique_id, ds")
  )

ds,unique_id,y
2018-01-31T00:00:00Z,M1,8900.0
2018-02-28T00:00:00Z,M1,8480.0
2018-03-31T00:00:00Z,M1,7870.0
2018-04-30T00:00:00Z,M1,7860.0
2018-05-31T00:00:00Z,M1,8990.0
2018-06-30T00:00:00Z,M1,10790.0
2018-07-31T00:00:00Z,M1,11820.0
2018-08-31T00:00:00Z,M1,10590.0
2018-09-30T00:00:00Z,M1,9430.0
2018-10-31T00:00:00Z,M1,9070.0


By default, `spark.sql.shuffle.partitions` is 200 regardless of cluster size, and `spark.task.cpus` is 1. With `groupBy("unique_id").applyInPandas(...)`, each group runs as one task on one core, and the number of concurrent tasks is capped by `spark.sql.shuffle.partitions`. So on a large cluster (e.g. 1000 cores), the default of 200 would leave 800 cores idle when you have many series. We therefore raise `spark.sql.shuffle.partitions` to `n` (the number of series) whenever `n` exceeds the current setting, so every series gets its own partition and all useful parallelism is realized. (When n is smaller than the number of cores, the leftover cores are an inherent limit of the workload — a single series can't be split across cores.)

In [0]:
# Get the current value of shuffle partitions
current = spark.conf.get("spark.sql.shuffle.partitions")

# If not set to 'auto' (serverless), convert to int; otherwise, use default 200
if current != "auto":
    current_val = int(current)
else:
    current_val = 200       

# If n is greater than the current value, update the shuffle partitions setting
if n > current_val:            
    spark.conf.set("spark.sql.shuffle.partitions", str(n))

Note that monthly forecasting requires the timestamp column to represent the last day of each month.

### Models
Let's configure a list of models we are going to apply to our time series for evaluation and forecasting. A comprehensive list of all supported models is available in [mmf_sa/models/README.md](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/README.md). Look for the models where `model_type: local`; these are the local models we import from [statsforecast](https://github.com/Nixtla/statsforecast) and [sktime](https://github.com/sktime/sktime). Check their documentations for the description of each model. 

In [0]:
active_models = [
    "StatsForecastBaselineWindowAverage",
    "StatsForecastBaselineSeasonalWindowAverage",
    "StatsForecastBaselineNaive",
    "StatsForecastBaselineSeasonalNaive",
    "StatsForecastAutoArima",
    "StatsForecastAutoETS",
    "StatsForecastAutoCES",
    "StatsForecastAutoTheta",
    "StatsForecastAutoTbats",
    "StatsForecastAutoMfles",
    "StatsForecastTSB",
    "StatsForecastADIDA",
    "StatsForecastIMAPA",
    "StatsForecastCrostonClassic",
    "StatsForecastCrostonOptimized",
    "StatsForecastCrostonSBA",
    "SKTimeProphet",
]

### Run MMF

Now, we can run the evaluation and forecasting using `run_forecast` function defined in [mmf_sa/models/__init__.py](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/__init__.py). Make sure to set `freq="M"` in `run_forecast` function.

In [0]:
run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.m4_monthly_train",
    scoring_data=f"{catalog}.{db}.m4_monthly_train",
    scoring_output=f"{catalog}.{db}.monthly_scoring_output",
    evaluation_output=f"{catalog}.{db}.monthly_evaluation_output",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="M",
    prediction_length=3,
    backtest_length=12,
    stride=1,
    metric="smape",
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/m4_monthly",
    use_case_name="m4_monthly",
)

Run quality checks


INFO:mmf_sa.data_quality_checks:Starting data quality checks...
INFO:mmf_sa.data_quality_checks:Initial dataset: 5940 records across 99 groups
INFO:mmf_sa.data_quality_checks:Running mandatory configuration checks...
INFO:mmf_sa.data_quality_checks:Running optional data quality checks...
INFO:mmf_sa.data_quality_checks:All groups passed data quality checks
INFO:mmf_sa.data_quality_checks:Data quality checks completed successfully. Final dataset: 5940 records across 99 groups


Finished quality checks
Starting evaluate_score
Starting evaluate_models
Started evaluating StatsForecastBaselineWindowAverage
  metric_name  metric_value
0       smape      0.086238
Finished evaluating StatsForecastBaselineWindowAverage
Started evaluating StatsForecastBaselineSeasonalWindowAverage
  metric_name  metric_value
0       smape           NaN
Finished evaluating StatsForecastBaselineSeasonalWindowAverage
Started evaluating StatsForecastBaselineNaive
  metric_name  metric_value
0       smape      0.096275
Finished evaluating StatsForecastBaselineNaive
Started evaluating StatsForecastBaselineSeasonalNaive
  metric_name  metric_value
0       smape      0.084039
Finished evaluating StatsForecastBaselineSeasonalNaive
Started evaluating StatsForecastAutoArima
  metric_name  metric_value
0       smape      0.063498
Finished evaluating StatsForecastAutoArima
Started evaluating StatsForecastAutoETS
  metric_name  metric_value
0       smape      0.062436
Finished evaluating StatsForec

'83517d0d-584f-43f6-b505-a442fccdf763'

### Evaluate
In `evaluation_output` table, the we store all evaluation results for all backtesting trials from all models.

In [0]:
display(
  spark.sql(f"""
    select * from {catalog}.{db}.monthly_evaluation_output 
    where unique_id = 'M1'
    order by unique_id, model, backtest_window_start_date
    """))

unique_id,backtest_window_start_date,metric_name,metric_value,forecast,actual,model_pickle,forecast_lower,forecast_upper,run_id,run_date,model,use_case,model_uri
M1,2022-01-31T00:00:00Z,smape,0.3209999384578449,"List(4789.336793953535, 4072.2877661859557, 3852.9680797753826)","List(5120.0, 5960.0, 6560.0)",gAWVpT0AAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-02-28T00:00:00Z,smape,0.40546480270987,"List(4113.86057834463, 3882.373774278028, 3486.6949020148204)","List(5960.0, 6560.0, 4900.0)",gAWVBj4AAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-03-31T00:00:00Z,smape,0.2591324529843621,"List(4100.030226402502, 3697.808825594929, 4359.218299745789)","List(6560.0, 4900.0, 4520.0)",gAWVbz4AAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-04-30T00:00:00Z,smape,0.13250268407057178,"List(3764.498066840988, 4429.999293336433, 6566.612353384014)","List(4900.0, 4520.0, 7370.0)",gAWV1D4AAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-05-31T00:00:00Z,smape,0.09227293103420119,"List(4678.976554021486, 6833.866039649894, 7656.945210122176)","List(4520.0, 7370.0, 9050.0)",gAWVNT8AAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-06-30T00:00:00Z,smape,0.14643452389789804,"List(6781.596626936858, 7601.065172022074, 6481.417315813868)","List(7370.0, 9050.0, 7780.0)",gAWVmj8AAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-07-31T00:00:00Z,smape,0.1376790531109447,"List(7697.2853190803835, 6583.268871503492, 4942.060295372989)","List(9050.0, 7780.0, 5380.0)",gAWVA0AAAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-08-31T00:00:00Z,smape,0.10982767101690703,"List(6856.136023316074, 5230.838965379249, 5602.069413018608)","List(7780.0, 5380.0, 4700.0)",gAWVZEAAAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,SKTimeProphet,m4_monthly,
M1,2022-09-30T00:00:00Z,smape,0.08038614293103229,"List(5404.16357698319, 5786.0354682252055, 4624.612510682135)","List(5380.0, 4700.0, 4490.0)",gAWVzUAAAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f

### Forecast
In `scoring_output` table, forecasts for each time series from each model are stored.

In [0]:
display(spark.sql(f"""
    select * from {catalog}.{db}.monthly_scoring_output 
    where unique_id = 'M1'
    order by unique_id, model, ds
    """))

unique_id,ds,y,model_pickle,forecast_lower,forecast_upper,run_id,run_date,use_case,model,model_uri
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4974.514306169162, 4659.995909433403, 4460.624593342521)",gAWVYUIAAAAAAACMHHNrdGltZS5mb3JlY2FzdGluZy5mYnByb3BoZXSUjAdQcm9waGV0lJOUKYGUfZQojARmcmVxlIwBTZSMD2FkZF9zZWFzb25hbGl0eZROjBRhZGRfY291bnRyeV9ob2xpZGF5c5ROjAZncm93dGiUjAZsaW4= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,SKTimeProphet,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(5096.531902036497, 5096.531902036497, 5096.531902036497)",gAWVgQwAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAVBRElEQZSTlCmBlH2UKIwFYWxpYXOUjAVBRElEQZSMFHByZWQ= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastADIDA,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4732.573163216812, 4654.769979177518, 4872.331808779956)",gAWVTGQAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAlBdXRvQVJJTUGUk5QpgZR9lCiMAWSUTowBRJROjAVtYXhfcJQ= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastAutoArima,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4841.742050170898, 4533.699951171875, 4561.9140625)",gAWVwxYAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAdBdXRvQ0VTlJOUKYGUfZQojA1zZWFzb25fbGVuZ3RolEsMjAU= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastAutoCES,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4996.233018106388, 4655.927940038661, 4690.571260237756)",gAWV1SsAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAdBdXRvRVRTlJOUKYGUfZQojA1zZWFzb25fbGVuZ3RolEsMjAU= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastAutoETS,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4659.570794222649, 4759.957476701542, 4851.841261865976)",gAWVWBwAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAlBdXRvTUZMRVOUk5QpgZR9lCiMDXNlYXNvbl9sZW5ndGiUSwc= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastAutoMfles,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(5327.045122005009, 5938.712694439512, 6181.269517137752)",gAWVPhkAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAlBdXRvVEJBVFOUk5QpgZR9lCiMDXNlYXNvbl9sZW5ndGiUXZQ= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastAutoTbats,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4801.26030136036, 4605.249231161135, 4665.008855892693)",gAWV6RUAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAlBdXRvVGhldGGUk5QpgZR9lCiMDXNlYXNvbl9sZW5ndGiUSww= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastAutoTheta,
M1,"List(2023-01-31T00:00:00Z, 2023-02-28T00:00:00Z, 2023-03-31T00:00:00Z)","List(4010.0, 4010.0, 4010.0)",gAWVjQwAAAAAAACMEnN0YXRzZm9yZWNhc3QuY29yZZSMDVN0YXRzRm9yZWNhc3SUk5QpgZR9lCiMBm1vZGVsc5RdlIwUc3RhdHNmb3JlY2FzdC5tb2RlbHOUjAVOYWl2ZZSTlCmBlH2UKIwFYWxpYXOUjAVOYWl2ZZSMFHByZWQ= (truncated),List(),List(),83517d0d-584f-43f6-b505-a442fccdf763,2026-09-29T07:17:17.752109Z,m4_monthly,StatsForecastBaselineNaive,
M1

Refer to the [notebook](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/post-evaluation-analysis.ipynb) for guidance on performing fine-grained model selection after running `run_forecast`.

### Delete Tables
Let's clean up the tables.

In [0]:
#display(spark.sql(f"delete from {catalog}.{db}.monthly_evaluation_output"))

In [0]:
#display(spark.sql(f"delete from {catalog}.{db}.monthly_scoring_output"))